# 🎬 SAM 2 (Segment Anything Model 2) 실습 노트북

> **논문**: *SAM 2: Segment Anything in Images and Videos* — Ravi et al., Meta AI (FAIR), 2024 ([arXiv:2408.00714](https://arxiv.org/abs/2408.00714))  
> **저자**: Nikhila Ravi, Valentin Gabeur, Yuan-Ting Hu, Ronghang Hu, Chaitanya Ryali, Tengyu Ma, Haitham Khedr, Roman Rädle, Chloe Rolland, Laura Gustafson, Eric Mintun, Junting Pan, Kalyan Vasudev Alwala, Nicolas Carion, Chao-Yuan Wu, Ross Girshick, Piotr Dollár, Christoph Feichtenhofer  
> **공식 코드**: <https://github.com/facebookresearch/sam2>

본 노트북은 **SAM 2** 논문의 내용을 단계별로 따라가며, **이미지뿐 아니라 비디오에서도** 점·박스 프롬프트로 객체를 분할(segmentation)하고 추적(tracking)하는 실습 노트북입니다.

이 노트북에서 다루는 내용:

1. SAM v1 → SAM 2 진화 한눈에 보기
2. **PVS** (Promptable Visual Segmentation) — 비디오로 확장된 새 태스크 정의
3. SAM 2 아키텍처 — **Hiera Image Encoder + Memory Attention + Memory Encoder + Memory Bank**
4. SA-V 데이터셋과 3단계 비디오 Data Engine
5. 코드 실습 ①: `SAM2ImagePredictor` (이미지 점/박스 프롬프트)
6. 코드 실습 ②: `SAM2AutomaticMaskGenerator` (이미지 자동 분할)
7. 코드 실습 ③: `SAM2VideoPredictor` (비디오 한 프레임에 프롬프트 → 전체 영상 propagation)
8. 코드 실습 ④: **다중 객체 추적** (여러 ID 동시 추적)
9. 코드 실습 ⑤: **보정 클릭 (refinement clicks)** 으로 마스크 다듬기
10. SAM 2 모델 크기 비교 (Hiera-T / S / B+ / L)

> ⚠️ **GPU 사용 권장**: Colab 상단 메뉴 **`런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU (T4 이상, 가능하면 L4/A100)`** 으로 설정하세요. Hiera-Large는 약 8-10GB VRAM을 사용합니다. PyTorch 2.5.1 이상이 필요합니다.


---
## 📖 0. SAM v1 → SAM 2: 무엇이 새로워졌나? (논문 한눈에 보기)

> *"We extend SAM to video by considering images as a video with a single frame. The model design is a simple transformer architecture with streaming memory for real-time video processing."* — Ravi et al., 2024

SAM 2는 단순히 "비디오 버전"이 아니라, **이미지를 1프레임짜리 비디오로 일반화**하여 **하나의 통일된 모델**로 이미지와 비디오를 모두 처리합니다. 이것이 핵심 디자인 결정입니다.

### 핵심 변화점

| 구분 | SAM v1 (2023) | **SAM 2 (2024)** |
|---|---|---|
| 입력 도메인 | 이미지만 | **이미지 + 비디오** (통합) |
| Task | Promptable Segmentation | **Promptable Visual Segmentation (PVS)** |
| Image Encoder | ViT-H/16 (MAE) | **Hiera** (multi-scale, MAE) |
| 시간 차원 처리 | 없음 | **Memory Attention + Memory Bank** |
| 학습 데이터 | SA-1B (11M 이미지, 1.1B 마스크) | + **SA-V** (51K 비디오, 643K masklets) |
| 이미지 속도 | 기준 | **6× 빠름** |
| 비디오 정확도 | — | **3× 적은 상호작용으로 더 정확** |
| 가장 큰 모델 | 636M (ViT-H) | 224M (Hiera-L) **— 더 작은데 더 빠르고 강함** |
| 다중 객체 추적 | 불가능 | **가능** (SAM2VideoPredictor) |

### 논문의 3대 기여 (논문 §1)

1. **Promptable Visual Segmentation Task (PVS)** — 비디오 어느 프레임의 점/박스/마스크 프롬프트라도 받아 **전체 비디오에 걸친 masklet(시공간 마스크)** 을 반환하는 일반화된 태스크
2. **SAM 2 모델** — Streaming memory를 가진 단일 transformer 아키텍처. 이미지에서 SAM보다 빠르고 정확
3. **SA-V 데이터셋** — 51K 비디오 / 642K masklets / 35.5M 마스크 (역대 최대 비디오 분할 데이터셋, SAM v1 대비 53배 많은 마스크)


---
## 🎯 1. Promptable Visual Segmentation Task (논문 §3)

### 1.1 PVS 정의

> *"We propose the promptable visual segmentation (PVS) task, which generalizes image segmentation to the video domain. The task takes as input points, boxes, or masks on any frame of a video to define a segment of interest and predict a spatio-temporal mask (i.e., a 'masklet')."*

**입력과 출력**:

- 입력: **비디오의 임의의 프레임**에 주어진 점(클릭)·박스·마스크 프롬프트
- 출력: **masklet** — 비디오 전체에 걸친 시공간(spatio-temporal) 마스크 시퀀스

### 1.2 핵심 아이디어 — Interactive video segmentation

비디오 작업의 핵심은 **점진적인 보정(refinement)** 입니다. 사용자가:

1. 1번 프레임에 **첫 클릭** → SAM 2가 그 프레임의 마스크를 생성하고 **나머지 프레임까지 propagate**
2. 어딘가에서 분할이 잘못된 프레임을 발견 → **그 프레임에서 보정 클릭(전경/배경)** 추가
3. SAM 2가 보정 정보까지 반영하여 **다시 propagate**

이 과정을 반복하면서 점점 정확한 masklet을 얻습니다. 논문에서는 사람과 모델이 상호작용하는 이 평가 프로토콜을 **interactive offline / online evaluation**이라고 부릅니다.

### 1.3 이미지는 곧 1프레임 비디오

> *"An image can be considered as a video with a single frame and we use the same model for both images and videos."*

이 단순한 일반화 덕분에 SAM 2는 SAM v1의 모든 능력(이미지 점/박스/박스+점/자동 마스크 생성)을 **그대로 보유**하면서 비디오 능력을 추가합니다.


---
## 🏗️ 2. SAM 2 아키텍처 (논문 §4)

![SAM 2 Architecture](https://github.com/facebookresearch/sam2/raw/main/assets/model_diagram.png)

*(이미지 출처: facebookresearch/sam2 공식 저장소)*

> *"For a given frame, the segmentation prediction is conditioned on the current prompt and/or on previously observed memories. Videos are processed in a streaming fashion with frames being consumed one at a time by the image encoder, which cross-attends to memories of the target object from previous frames."*

SAM 2는 **5개 주요 컴포넌트**로 구성됩니다:

1. **Image Encoder** (Hiera) — 프레임별 임베딩 추출 (스트리밍)
2. **Memory Attention** — 현재 프레임 임베딩을 과거 메모리에 cross-attend
3. **Prompt Encoder** — SAM v1과 동일
4. **Mask Decoder** — 마스크 + IoU + object presence 예측
5. **Memory Encoder + Memory Bank** — 과거 예측을 압축해 저장


### 2.1 Hiera Image Encoder (논문 §4.1)

> *"For the image encoder we use a pre-trained Hiera image encoder which is hierarchical, allowing us to use multi-scale features during decoding."*

SAM v1의 **ViT-H/16** (단일 스케일)을 **Hiera** (Hierarchical Vision Transformer, MAE pretrained) 로 교체했습니다.

**Hiera의 특징**:

- **4단계 hierarchical 구조** — stride **4, 8, 16, 32**의 multi-scale feature를 출력
- Stride 16, 32 (저해상도/coarse) → Memory attention 입력으로 사용
- Stride 4, 8 (고해상도/fine) → Mask decoder의 **업샘플링 layer**에 직접 연결 (skip connection)
- Windowed absolute positional embedding 사용 (RPB 대신)

**왜 이렇게 바꿨나?**

| 기준 | ViT-H (SAM v1) | Hiera-L (SAM 2) |
|---|---|---|
| 파라미터 | 636M | **224M** |
| Multi-scale | ❌ 단일 스케일 | ✅ 4-stage |
| Fine 디테일 보존 | 16× downsample만 가능 | **4× / 8× feature 직접 활용** |
| Image 추론 속도 (A100) | 기준 | **약 6× 빠름** |

작은 객체나 경계가 복잡한 객체를 더 정밀하게 잡으면서도 **더 작고 더 빠른** 모델이 만들어진 비결입니다.


### 2.2 Prompt Encoder (논문 §4.2 — SAM v1과 동일)

> *"Our prompt encoder, identical to SAM's, can be prompted by clicks (positive or negative), bounding boxes, or masks to define the extent of the object in a given frame."*

- **Points**: positional encoding + 학습된 전경/배경 임베딩
- **Boxes**: 좌상단·우하단 점의 positional encoding + 코너 임베딩
- **Mask**: 합성곱으로 다운샘플링 후 image embedding에 더함

> 💡 SAM v1과 거의 동일. 단, 비디오에서는 **임의의 프레임에 prompt를 줄 수 있고**, 시간 인덱스가 추가됩니다.


### 2.3 Memory Attention (논문 §4.3) ⭐ 핵심 신규

> *"The role of memory attention is to condition the current frame features on the past frames features and predictions as well as on any new prompts. We stack L=4 transformer blocks, the first one taking the image encoding from the current frame as input. Each block performs self-attention, followed by cross-attention to memories of (prompted/unprompted) frames and object pointers, ... followed by an MLP."*

이것이 **SAM 2의 핵심 신규 구성요소**입니다. 현재 프레임의 임베딩이 **과거의 메모리에 cross-attend**하여 시간적 일관성을 확보합니다.

**구조**:

- **L = 4 transformer blocks**
- 각 블록: **Self-attention** (현재 프레임) → **Cross-attention** (과거 메모리) → **MLP**
- Vanilla attention 사용 (FlashAttention 등 효율적 커널 활용 가능)
- **2D RoPE** (Rotary Positional Embedding) 사용 (object pointer 토큰은 제외)

**무엇에 cross-attend 하나?**

1. **Spatial memory features**: 과거 프레임들의 마스크 + 임베딩을 결합한 공간 맵 (memory encoder가 생성)
2. **Object pointers**: 각 과거 프레임의 마스크 디코더 출력 토큰을 가벼운 벡터로 압축한 것 (객체의 "고수준 의미"를 빠르게 참조)


### 2.4 Memory Encoder + Memory Bank (논문 §4.4) ⭐ 핵심 신규

> *"The memory encoder generates a memory by downsampling the output mask using a convolutional module and summing it element-wise with the unconditioned frame embedding from the image encoder ... The memory bank retains information about past predictions for the target object in the video by maintaining a FIFO queue of memories of up to N=6 recent frames and stores information from prompts in a FIFO queue of up to M=1 prompted frames."*

#### Memory Encoder

각 프레임의 **예측 마스크**를 다음과 같이 인코딩하여 메모리로 저장합니다:

1. 마스크를 conv로 다운샘플링
2. 같은 프레임의 image embedding과 **원소별 합**
3. 가벼운 convolutional layer로 fuse

#### Memory Bank — **두 개의 FIFO 큐**

| 큐 | 크기 | 저장 대상 |
|---|---|---|
| **Unprompted recent frames** | **N = 6** | 가장 최근에 처리된 (사용자 클릭이 없었던) 프레임들 |
| **Prompted frames** | **M = 1** | 사용자가 직접 클릭/박스를 준 프레임 (예: 시작 프레임) |
| **Object pointers** | 모든 과거 프레임 | mask decoder의 output token (벡터) |

이 구조는 **고정된 메모리 크기로 임의 길이의 비디오를 스트리밍 처리**할 수 있게 해줍니다. 가장 최근 6 프레임의 풍부한 공간 정보 + 사용자가 명시적으로 준 1 프레임의 prompt 정보 + 모든 과거 프레임의 가벼운 object pointer를 동시에 보는 것이죠.

#### 시간적 위치 정보

> *"We embed temporal position information into the memories of N recent frames, allowing the model to represent short-term object motion, but not into those of prompted frames, since the training signal from prompted frames is more sparse and it is more difficult to generalize to inference settings ... at different temporal ranges."*

- N=6 최근 프레임 → **temporal positional embedding 추가** (단기 움직임 학습)
- 사용자 prompted 프레임 → temporal embedding **없음** (어느 시점이든 일반화 가능하도록)


### 2.5 Mask Decoder — 이미지 vs 비디오 (논문 §4.5)

> *"For ambiguous prompts (i.e., a single click) where there may be multiple compatible target objects, we predict multiple masks. This design is important to ensure that the model outputs valid masks. In videos, ambiguities can extend across video frames, and the model predicts multiple masks on each frame. ... we only propagate the one with the highest predicted IoU."*

기본 구조는 SAM v1과 같지만 **비디오용 추가 출력**이 있습니다:

| 출력 | SAM v1 | SAM 2 |
|---|---|---|
| **마스크 (1 또는 3개)** | ✅ | ✅ |
| **예측 IoU** | ✅ | ✅ |
| **Occlusion(가려짐) 예측** | ❌ | ✅ **신규** — 현재 프레임에 객체가 있는지(visible) 여부 |
| **Fine-resolution skip** | ❌ | ✅ Hiera stride 4/8을 업샘플링 layer에 직접 결합 |

**Occlusion head**의 역할: 비디오에서는 객체가 일시적으로 가려졌다가 다시 나타납니다. SAM 2는 **객체가 보이지 않는 프레임에서는 빈 마스크를 출력**할 수 있어야 하므로, "현재 프레임에 객체가 존재하는가"를 별도로 예측합니다.


---
## 📚 3. SA-V 데이터셋과 비디오 Data Engine (논문 §5)

### 3.1 SA-V — 역대 최대 비디오 분할 데이터셋

| 항목 | 수치 |
|---|---|
| 비디오 수 | **50.9K** |
| Masklet 수 | **642.6K** (의미 있는 객체 트랙) |
| 총 마스크 수 | **35.5M** |
| SAM v1의 SA-1B 마스크 수와 비교 | **약 53배 많은 마스크** ⚡ |
| 평균 비디오 길이 | 14초 (24 FPS, 50% 이상이 100 프레임 이상) |
| 클래스 | open-world (closed vocabulary 없음) |
| 라이선스 | SA-V Research License (개인정보 블러 처리) |

### 3.2 Data Engine — 3 단계 (논문 §5.2)

#### Phase 1: SAM per-frame
> 비디오의 **각 프레임을 SAM v1으로 따로** 분할 → 어노테이터가 수동 보정 → 매우 느림 (37.8s/frame).
> 결과: 16K masklets / 1.4K videos

#### Phase 2: SAM + SAM 2 Mask
> 초기 SAM 2 (마스크 prompt만 지원)가 등장. 첫 프레임은 SAM이, 이후 프레임은 SAM 2가 mask 기반으로 propagate. 5.1배 빠름.
> 결과: 63.5K masklets

#### Phase 3: SAM 2 (full)
> 완성된 SAM 2가 점·박스·마스크 어떤 프롬프트든 모든 프레임에서 받음. 8.4배 빠름 (4.5s/frame).
> 결과: SA-V 본체 (642K masklets)

> 💡 SAM v1의 data engine과 동일한 철학(model-in-the-loop) — **모델이 좋아질수록 어노테이션이 빨라지고**, 그 데이터로 모델이 또 좋아집니다.


---
## ⚙️ 4. 환경 설정 (Colab)

### 4.1 GPU 확인


In [ ]:
# GPU 확인 (런타임 → 런타임 유형 변경 → GPU)
!nvidia-smi


### 4.2 SAM 2 설치

> ⚠️ **주의**: SAM 2는 **PyTorch 2.5.1 이상**을 요구합니다. Colab의 기본 PyTorch가 더 낮으면 `pip install`이 자동으로 업그레이드합니다. 또한 custom CUDA extension을 빌드하는데, 빌드가 실패해도 추론은 동작합니다 (post-processing만 제한).


In [ ]:
# PyTorch 버전 확인
import torch
print('Current PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)

# PyTorch가 너무 낮으면 업그레이드 필요 (보통 Colab은 2.5+이므로 자동)


In [ ]:
# 환경 설정 (이전 차시 패턴 동일)
import warnings, logging
import math, random, copy, time, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 디바이스: {device}")

# 한글 폰트
import subprocess
try:
    subprocess.run(['apt-get','-qq','install','fonts-nanum'], check=False)
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print(f"폰트 생략: {e}")
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'dejavusans'

import timm
print(f"✅ timm 버전: {timm.__version__}")
print(f"✅ torchvision 버전: {datasets.__name__.split('.')[0]} 확인 완료")
print("✅ 환경 설정 완료")


In [ ]:
# SAM 2 공식 저장소 clone & install
%cd /content
![ -d sam2 ] || git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -q -e .
!pip install -q matplotlib opencv-python pillow
print('SAM 2 설치 완료')


### 4.3 SAM 2.1 체크포인트 다운로드

논문에서는 **Hiera-T / S / B+ / L** 4가지 사이즈를 공개했고, **SAM 2.1**은 2024년 9월에 발표된 개선 체크포인트입니다. 본 노트북에서는 **base_plus** (~80M, 64 FPS)로 진행합니다. 더 강한 결과가 필요하면 `large`로 바꾸세요.


In [ ]:
# SAM 2.1 체크포인트 다운로드 (base_plus, ~330MB)
import os
os.makedirs('/content/sam2/checkpoints', exist_ok=True)
%cd /content/sam2/checkpoints

# 4종 체크포인트 (필요한 것만 받으세요)
BASE_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824'
!wget -q --show-progress {BASE_URL}/sam2.1_hiera_base_plus.pt -O sam2.1_hiera_base_plus.pt

# (선택) Tiny / Small / Large도 받고 싶으면 주석 해제
# !wget -q --show-progress {BASE_URL}/sam2.1_hiera_tiny.pt -O sam2.1_hiera_tiny.pt
# !wget -q --show-progress {BASE_URL}/sam2.1_hiera_small.pt -O sam2.1_hiera_small.pt
# !wget -q --show-progress {BASE_URL}/sam2.1_hiera_large.pt -O sam2.1_hiera_large.pt

!ls -lh *.pt
%cd /content/sam2


### 4.4 예제 자산 다운로드

이미지는 SAM v1 노트북과 동일한 truck 이미지를 사용하고, 비디오는 공식 SAM 2 예제의 **bedroom** JPEG 시퀀스를 사용합니다.


In [ ]:
# 이미지 예제
!mkdir -p /content/assets
!wget -q https://raw.githubusercontent.com/facebookresearch/sam2/main/notebooks/images/truck.jpg -O /content/assets/truck.jpg

# 비디오 예제 (bedroom JPEG 시퀀스, 200 프레임)
!mkdir -p /content/assets/bedroom
import urllib.request
BASE = 'https://raw.githubusercontent.com/facebookresearch/sam2/main/notebooks/videos/bedroom'

# 처음 60 프레임만 받기 (전체는 200 프레임)
N_FRAMES = 60
for i in range(N_FRAMES):
    fname = f'{i:05d}.jpg'
    url = f'{BASE}/{fname}'
    out = f'/content/assets/bedroom/{fname}'
    if not os.path.exists(out):
        try:
            urllib.request.urlretrieve(url, out)
        except Exception as e:
            print(f'{fname}: {e}')
            break

import glob
frames = sorted(glob.glob('/content/assets/bedroom/*.jpg'))
print(f'다운로드한 프레임 수: {len(frames)}')


---
## 🛠️ 5. 공통 유틸리티

분할 결과 시각화를 위한 헬퍼 함수들을 정의합니다 (공식 노트북과 동일).


In [ ]:
import os, sys
sys.path.insert(0, '/content/sam2')

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# bfloat16 자동 캐스팅 활성화 (SAM 2 권장)
torch.autocast(device_type='cuda', dtype=torch.bfloat16).__enter__()
if torch.cuda.get_device_properties(0).major >= 8:
    # Ampere(A100) 이상: TF32 활성화
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# 시각화 함수들 (공식 SAM 2 노트북 헬퍼와 동일)
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    elif obj_id is not None:
        # 객체 ID별 고정 컬러
        cmap = plt.get_cmap('tab10')
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array([*cmap(cmap_idx)[:3], 0.6])
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=200):
    pos = coords[labels == 1]
    neg = coords[labels == 0]
    ax.scatter(pos[:, 0], pos[:, 1], color='green', marker='*',
               s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg[:, 0], neg[:, 1], color='red', marker='*',
               s=marker_size, edgecolor='white', linewidth=1.25)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2]-box[0], box[3]-box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='lime',
                               facecolor=(0,0,0,0), lw=2))

def show_anns(anns, borders=True):
    if len(anns) == 0: return
    sorted_anns = sorted(anns, key=lambda x: x['area'], reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)
    img = np.ones((sorted_anns[0]['segmentation'].shape[0],
                   sorted_anns[0]['segmentation'].shape[1], 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.5]])
        img[m] = color_mask
    ax.imshow(img)


---
## 🖼️ 6. [이미지] `SAM2ImagePredictor` — SAM v1과 같은 인터페이스, 6배 빠른 속도

> *"SAM 2 has all the capabilities of SAM on static images, and we provide image prediction APIs that closely resemble SAM for image use cases."*

이미지 사용은 SAM v1과 거의 동일합니다. `set_image()` → `predict()` 패턴 그대로입니다.


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import cv2

CHECKPOINT = '/content/sam2/checkpoints/sam2.1_hiera_base_plus.pt'
MODEL_CFG = 'configs/sam2.1/sam2.1_hiera_b+.yaml'

# 모델 로드
sam2_model = build_sam2(MODEL_CFG, CHECKPOINT, device='cuda')
image_predictor = SAM2ImagePredictor(sam2_model)

# 파라미터 수
n_params = sum(p.numel() for p in sam2_model.parameters())
print(f'SAM 2.1 hiera_base_plus 파라미터: {n_params/1e6:.1f}M')

# SAM 2 모델의 컴포넌트 구조 살펴보기 (논문의 5개 모듈)
print()
print('=== SAM 2 모델 컴포넌트 ===')
for name, module in sam2_model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f'  {name:30s} | {type(module).__name__:30s} | {n/1e6:7.1f}M')


In [ ]:
# 트럭 이미지 로드
image = Image.open('/content/assets/truck.jpg').convert('RGB')
image_np = np.array(image)
print('Image shape:', image_np.shape)

plt.figure(figsize=(10, 8))
plt.imshow(image_np)
plt.title('Input image')
plt.axis('off')
plt.show()


### 6.1 단일 점 프롬프트 + multimask output

SAM v1과 동일하게 단일 점에서는 모호성이 크므로 **3개의 후보 마스크 (whole / part / subpart)** 가 나옵니다.


In [ ]:
import time

# 이미지 임베딩 캐싱 (Hiera image encoder를 1회 실행)
t0 = time.time()
image_predictor.set_image(image_np)
t1 = time.time()
print(f'set_image() 소요: {(t1-t0)*1000:.1f} ms  ← SAM v1보다 약 6배 빠름')

# 단일 전경 점
input_point = np.array([[500, 375]])
input_label = np.array([1])

masks, scores, logits = image_predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,    # 3개 후보 출력
)
print('masks.shape =', masks.shape, ' (3개 후보)')
print('scores =', scores)


In [ ]:
# 3개 후보 마스크 시각화
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for i, (mask, score) in enumerate(zip(masks, scores)):
    axes[i].imshow(image_np)
    show_mask(mask, axes[i])
    show_points(input_point, input_label, axes[i])
    axes[i].set_title(f'Mask {i+1} (IoU pred {score:.3f})', fontsize=14)
    axes[i].axis('off')
plt.tight_layout()
plt.show()


### 6.2 박스 + 점 결합 프롬프트


In [ ]:
# 박스 + 배경 점 결합 (SAM v1과 동일한 API)
input_box = np.array([425, 600, 700, 875])
input_point = np.array([[575, 750]])
input_label = np.array([0])  # 배경

masks, scores, _ = image_predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    box=input_box[None, :],
    multimask_output=False,
)

plt.figure(figsize=(10, 8))
plt.imshow(image_np)
show_mask(masks[0], plt.gca())
show_box(input_box, plt.gca())
show_points(input_point, input_label, plt.gca())
plt.title(f'Box + 배경 점 (score {scores[0]:.3f})')
plt.axis('off')
plt.show()


---
## 🌐 7. [이미지] `SAM2AutomaticMaskGenerator` — 전체 이미지 자동 분할

> *"SAM 2 also supports automatic mask generation on images just like SAM."*

SAM v1과 동일하게 grid 점 방식으로 이미지 전체의 모든 객체를 자동으로 분할합니다. Hiera 백본 덕분에 더 빠릅니다.


In [ ]:
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=32,
    points_per_batch=64,
    pred_iou_thresh=0.7,
    stability_score_thresh=0.92,
    stability_score_offset=0.7,
    crop_n_layers=0,
    box_nms_thresh=0.7,
    min_mask_region_area=25.0,
    use_m2m=True,                # SAM 2 신규: mask-to-mask refinement
)

t0 = time.time()
masks_auto = mask_generator.generate(image_np)
t1 = time.time()
print(f'자동 분할: {len(masks_auto)} masks, {t1-t0:.1f}s')

# 마스크 dict 키 살펴보기 (SAM v1과 거의 동일)
print('마스크 dict keys:', list(masks_auto[0].keys()))


In [ ]:
plt.figure(figsize=(12, 10))
plt.imshow(image_np)
show_anns(masks_auto)
plt.axis('off')
plt.title(f'Automatic mask generation — {len(masks_auto)} masks')
plt.show()


---
## 🎥 8. [비디오] `SAM2VideoPredictor` — SAM 2의 진짜 강점 ⭐

이제 SAM 2의 가장 중요한 기능, **비디오 분할 + 추적**을 실습합니다. 워크플로는 다음 3단계입니다:

```
1. predictor.init_state(video_path)                  # inference state 초기화
2. predictor.add_new_points_or_box(...)              # 임의의 프레임에 prompt 추가
3. for frame_idx, obj_ids, masks in predictor.propagate_in_video(state):
       # 전체 비디오에 걸쳐 자동으로 masklet 전파
```

내부에서는 논문의 **streaming memory pipeline** (memory attention + memory encoder + memory bank)이 동작합니다.

### 8.1 비디오 predictor 초기화


In [ ]:
from sam2.build_sam import build_sam2_video_predictor

video_predictor = build_sam2_video_predictor(MODEL_CFG, CHECKPOINT, device='cuda')

# 비디오 디렉토리 (JPEG 프레임 시퀀스)
VIDEO_DIR = '/content/assets/bedroom'

# 프레임 파일 목록 (이름순 정렬)
frame_names = sorted([
    p for p in os.listdir(VIDEO_DIR)
    if os.path.splitext(p)[-1].lower() in ['.jpg', '.jpeg']
])
print(f'총 프레임 수: {len(frame_names)}')

# inference state 초기화 (Hiera로 모든 프레임 임베딩을 미리 계산)
inference_state = video_predictor.init_state(video_path=VIDEO_DIR)
print('init_state 완료')


In [ ]:
# 첫 프레임 미리보기
plt.figure(figsize=(10, 6))
plt.imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[0])))
plt.title(f'Frame 0: {frame_names[0]}')
plt.axis('off')
plt.show()


### 8.2 첫 프레임에 점 프롬프트 — 객체 1개 등록

`add_new_points_or_box`로 **임의의 프레임**, **임의의 객체 ID**에 prompt를 추가합니다.

- `frame_idx`: 프롬프트를 줄 프레임 번호 (0부터 시작)
- `obj_id`: 객체에 할당할 고유 ID (정수, 임의로 정해도 OK)
- `points` + `labels`: 점 좌표와 fg/bg 라벨

함수는 **해당 프레임의 즉시 예측 마스크**를 반환합니다. (전파는 아직 안 함)


In [ ]:
# 첫 프레임에서 객체 1 (예: 의자)에 전경 점 1개
ann_frame_idx = 0
ann_obj_id = 1     # 객체 ID (정수, 우리가 정함)

# 의자에 점 찍기 (좌표는 이미지에 따라 조절)
points = np.array([[210, 350]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)  # 1 = foreground

frame_idx, obj_ids, masks = video_predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)
print(f'Prompt 추가 — frame {frame_idx}, obj_ids={obj_ids}')
print(f'반환된 마스크 logits shape: {masks.shape}')

# 즉시 시각화
plt.figure(figsize=(10, 6))
plt.imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[ann_frame_idx])))
show_points(points, labels, plt.gca())
mask = (masks[0] > 0.0).cpu().numpy()
show_mask(mask, plt.gca(), obj_id=ann_obj_id)
plt.title(f'Frame {ann_frame_idx}: 첫 클릭 직후 마스크')
plt.axis('off')
plt.show()


### 8.3 `propagate_in_video` — Memory를 이용해 전체 비디오로 전파

이제 **memory attention + memory bank**가 동작하는 핵심 부분입니다. SAM 2는 첫 프레임의 마스크를 **memory encoder로 인코딩 → memory bank에 저장 → 다음 프레임에서 cross-attend**하는 식으로 스트리밍 처리합니다.


In [ ]:
# 전체 비디오에 걸쳐 propagate (자동)
video_segments = {}  # frame_idx -> {obj_id: mask}

t0 = time.time()
for out_frame_idx, out_obj_ids, out_mask_logits in video_predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        oid: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, oid in enumerate(out_obj_ids)
    }
t1 = time.time()
print(f'전체 {len(video_segments)} 프레임 propagate 완료, {t1-t0:.1f}s')
print(f'FPS: {len(video_segments)/(t1-t0):.1f}')


In [ ]:
# 추적 결과를 일정 간격으로 샘플링하여 시각화
sample_step = max(1, len(frame_names) // 6)
sample_indices = list(range(0, len(frame_names), sample_step))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, fi in zip(axes.flatten(), sample_indices):
    img = Image.open(os.path.join(VIDEO_DIR, frame_names[fi]))
    ax.imshow(img)
    if fi in video_segments:
        for oid, m in video_segments[fi].items():
            show_mask(m, ax, obj_id=oid)
    ax.set_title(f'Frame {fi}')
    ax.axis('off')
plt.tight_layout()
plt.show()


---
## 👥 9. [비디오] 다중 객체 추적 — 여러 ID 동시 처리

> SAM 2.1 (2024.12 업데이트)의 새 `SAM2VideoPredictor`는 **객체별로 독립적인 추론**을 지원합니다. 같은 비디오에서 여러 객체를 각자 다른 ID로 등록하고 동시에 추적할 수 있습니다.

먼저 inference state를 리셋하고 객체 2개를 등록해봅시다.


In [ ]:
# state 리셋
video_predictor.reset_state(inference_state)

# 첫 프레임에 객체 1 (의자)
points1 = np.array([[210, 350]], dtype=np.float32)
labels1 = np.array([1], dtype=np.int32)
_, _, _ = video_predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=0, obj_id=1,
    points=points1, labels=labels1,
)

# 첫 프레임에 객체 2 (다른 영역, 예: 다른 가구)
# 좌표는 이미지에 맞게 조절
points2 = np.array([[450, 300]], dtype=np.float32)
labels2 = np.array([1], dtype=np.int32)
_, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=0, obj_id=2,
    points=points2, labels=labels2,
)

print(f'등록된 객체 ID들: {out_obj_ids}')

# 첫 프레임에 두 객체 마스크 시각화
plt.figure(figsize=(10, 6))
plt.imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[0])))
show_points(points1, labels1, plt.gca())
show_points(points2, labels2, plt.gca())
for i, oid in enumerate(out_obj_ids):
    m = (out_mask_logits[i] > 0.0).cpu().numpy()
    show_mask(m, plt.gca(), obj_id=oid)
plt.title('Frame 0: 객체 2개 동시 등록')
plt.axis('off')
plt.show()


In [ ]:
# 다시 전체 비디오로 propagate
video_segments = {}
for out_frame_idx, out_obj_ids, out_mask_logits in video_predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        oid: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, oid in enumerate(out_obj_ids)
    }
print(f'다중 객체 propagate 완료 ({len(video_segments)} frames)')

# 시각화
sample_indices = list(range(0, len(frame_names), max(1, len(frame_names)//6)))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, fi in zip(axes.flatten(), sample_indices):
    ax.imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[fi])))
    for oid, m in video_segments[fi].items():
        show_mask(m, ax, obj_id=oid)
    ax.set_title(f'Frame {fi}')
    ax.axis('off')
plt.tight_layout()
plt.show()


---
## ✏️ 10. [비디오] 보정 클릭 (Refinement Clicks) — 중간 프레임에서 마스크 다듬기

논문의 핵심 평가 시나리오 중 하나입니다.

> *"Interactive offline evaluation: clicks are added by an annotator on incorrectly tracked frames after watching all the predictions in the video."*

**시나리오**:
1. 첫 프레임에서 객체 1개 등록 → propagate
2. 어딘가 중간 프레임에서 분할이 잘못된 것을 발견
3. **그 프레임에 보정 클릭(전경 또는 배경)** 추가
4. **다시 propagate** → 보정된 결과 획득

memory bank의 "prompted frames" 큐 (M=1)에 새 prompt가 저장되어, 이후 propagation 시 양방향으로 참조됩니다.


In [ ]:
# 시나리오: 의자만 추적 시작 → 중간 프레임에서 부족한 부분에 fg 점 추가

video_predictor.reset_state(inference_state)

# Step 1: 첫 프레임 클릭
points = np.array([[210, 350]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)
video_predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=0, obj_id=1,
    points=points, labels=labels,
)

# 1차 propagate
first_segs = {}
for fi, oids, logits in video_predictor.propagate_in_video(inference_state):
    first_segs[fi] = {oid: (logits[i] > 0.0).cpu().numpy() for i, oid in enumerate(oids)}

print(f'1차 propagate 완료 ({len(first_segs)} frames)')


In [ ]:
# 가장 어려워 보이는 중간 프레임 하나 골라보기
refine_frame = min(len(frame_names) - 1, 30)

plt.figure(figsize=(10, 6))
plt.imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[refine_frame])))
for oid, m in first_segs[refine_frame].items():
    show_mask(m, plt.gca(), obj_id=oid)
plt.title(f'1차 결과 — Frame {refine_frame} (여기에 보정 클릭 추가 예정)')
plt.axis('off')
plt.show()


In [ ]:
# Step 2: 그 프레임에 보정 클릭 (전경 점 1개 추가)
# 좌표는 결과를 보고 조절. 여기서는 예시 좌표.
refine_points = np.array([[250, 400]], dtype=np.float32)
refine_labels = np.array([1], dtype=np.int32)  # 전경

_, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=refine_frame,
    obj_id=1,                # 같은 객체에 추가
    points=refine_points,
    labels=refine_labels,
)
print(f'보정 클릭 추가 (frame {refine_frame})')

# 그 프레임 즉시 결과
plt.figure(figsize=(10, 6))
plt.imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[refine_frame])))
show_points(refine_points, refine_labels, plt.gca())
m = (out_mask_logits[0] > 0.0).cpu().numpy()
show_mask(m, plt.gca(), obj_id=1)
plt.title(f'보정 클릭 직후 Frame {refine_frame}')
plt.axis('off')
plt.show()


In [ ]:
# Step 3: 보정 정보를 반영해 다시 propagate
refined_segs = {}
for fi, oids, logits in video_predictor.propagate_in_video(inference_state):
    refined_segs[fi] = {oid: (logits[i] > 0.0).cpu().numpy() for i, oid in enumerate(oids)}

# Before vs After 비교
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
compare_indices = [0, refine_frame, min(len(frame_names)-1, refine_frame+15)]

for col, fi in enumerate(compare_indices):
    # 위: 1차
    axes[0, col].imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[fi])))
    if fi in first_segs:
        for oid, m in first_segs[fi].items():
            show_mask(m, axes[0, col], obj_id=oid)
    axes[0, col].set_title(f'1차 (보정 전) — Frame {fi}')
    axes[0, col].axis('off')

    # 아래: 보정 후
    axes[1, col].imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[fi])))
    if fi in refined_segs:
        for oid, m in refined_segs[fi].items():
            show_mask(m, axes[1, col], obj_id=oid)
    axes[1, col].set_title(f'보정 후 — Frame {fi}')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()


---
## ⚖️ 11. 모델 크기 비교 — Hiera-T / S / B+ / L

SAM 2.1 체크포인트 4종의 공식 벤치마크입니다 (논문 + GitHub README).

| 모델 | 파라미터 | A100 FPS | SA-V test (J&F) | MOSE val (J&F) | LVOS v2 (J&F) |
|---|---|---|---|---|---|
| hiera_tiny | **38.9M** | **91.2** | 76.5 | 71.8 | 77.3 |
| hiera_small | 46.0M | 84.8 | 76.6 | 73.5 | 78.3 |
| hiera_base_plus | 80.8M | 64.1 | 78.2 | 73.7 | 78.2 |
| hiera_large | **224.4M** | 39.5 | **79.5** | **74.6** | **80.6** |

**J&F** = Jaccard (IoU) + F-measure (boundary) 평균, 비디오 분할의 표준 지표.

비교 포인트:

- **SAM v1의 ViT-H가 636M**이었던 것에 비해 **SAM 2.1 Large는 224M**로 **2.8배 작음**
- 그럼에도 이미지 정확도는 더 높고, 비디오는 새로 가능
- Colab에서 빠른 데모용이라면 **Tiny / Small**이 충분히 강력


In [ ]:
# 메모리 정리
del video_predictor, image_predictor, sam2_model, mask_generator
torch.cuda.empty_cache()

# (선택) Tiny 체크포인트도 받아서 속도 비교
import os
TINY_CKPT = '/content/sam2/checkpoints/sam2.1_hiera_tiny.pt'
if not os.path.exists(TINY_CKPT):
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt -O {TINY_CKPT}

# Tiny 로 동일 이미지 추론
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_tiny = build_sam2('configs/sam2.1/sam2.1_hiera_t.yaml', TINY_CKPT, device='cuda')
predictor_tiny = SAM2ImagePredictor(sam2_tiny)
n = sum(p.numel() for p in sam2_tiny.parameters())
print(f'Tiny 파라미터: {n/1e6:.1f}M')

image = Image.open('/content/assets/truck.jpg').convert('RGB')
image_np = np.array(image)

t0 = time.time(); predictor_tiny.set_image(image_np); t1 = time.time()
print(f'Tiny set_image: {(t1-t0)*1000:.1f}ms')

masks, scores, _ = predictor_tiny.predict(
    point_coords=np.array([[500, 375]]),
    point_labels=np.array([1]),
    multimask_output=True,
)
best = int(np.argmax(scores))

plt.figure(figsize=(8, 6))
plt.imshow(image_np)
show_mask(masks[best], plt.gca())
show_points(np.array([[500, 375]]), np.array([1]), plt.gca())
plt.title(f'Tiny 결과 (best of 3, IoU pred {scores[best]:.3f})')
plt.axis('off')
plt.show()

del predictor_tiny, sam2_tiny
torch.cuda.empty_cache()


---
## 🎓 12. 정리 및 다음 단계

### 이 노트북에서 배운 것

| 논문 섹션 | 키워드 | 실습 코드 |
|---|---|---|
| §3 PVS Task | Promptable Visual Segmentation, masklet | — |
| §4.1 Image Encoder | Hiera (multi-scale, MAE pretrained), stride 4/8/16/32 | `build_sam2`로 자동 로드 |
| §4.2 Prompt Encoder | SAM v1과 동일 (point/box/mask) | `add_new_points_or_box` |
| §4.3 Memory Attention | L=4 transformer, self-attn → cross-attn → MLP, 2D RoPE | `propagate_in_video` 내부 |
| §4.4 Memory Bank | FIFO 큐 (N=6 unprompted + M=1 prompted) + object pointers | `inference_state` |
| §4.5 Mask Decoder | + Occlusion head, Hiera fine-resolution skip | 자동 |
| §5 SA-V Dataset | 51K 비디오, 643K masklets, 35.5M 마스크 | — |
| 이미지 사용 | `SAM2ImagePredictor`.predict (SAM v1 호환 API) | §6 |
| 자동 마스크 | `SAM2AutomaticMaskGenerator` | §7 |
| 비디오 사용 | `init_state` → `add_new_points_or_box` → `propagate_in_video` | §8-9 |
| 다중 객체 | obj_id별 독립 추적 (SAM 2.1 신규) | §9 |
| 보정 클릭 | 중간 프레임에 prompt 추가 후 재propagate | §10 |

### SAM v1 vs SAM 2 — 최종 정리

| | **SAM v1** (2023) | **SAM 2** (2024) |
|---|---|---|
| 입력 | 이미지 | **이미지 + 비디오** |
| Backbone | ViT-H 636M | Hiera-L 224M |
| 이미지 속도 | 기준 | **6× 빠름** |
| 시간 정보 | 없음 | **Memory Attention + Memory Bank** |
| 추적 | 불가능 | **임의 길이 비디오 streaming** |
| 다중 객체 | 불가능 | **가능 (per-object inference)** |
| 데이터셋 | SA-1B (1.1B 마스크) | + SA-V (35.5M 마스크) |
| 최신 체크포인트 | sam_vit_h | **sam2.1_hiera_large** |

### 다음 단계

- 공식 데모: <https://sam2.metademolab.com/demo>
- 공식 예제 노트북:
  - [image_predictor_example.ipynb](https://github.com/facebookresearch/sam2/blob/main/notebooks/image_predictor_example.ipynb)
  - [automatic_mask_generator_example.ipynb](https://github.com/facebookresearch/sam2/blob/main/notebooks/automatic_mask_generator_example.ipynb)
  - [video_predictor_example.ipynb](https://github.com/facebookresearch/sam2/blob/main/notebooks/video_predictor_example.ipynb)
- 학습/파인튜닝: [`training/README.md`](https://github.com/facebookresearch/sam2/blob/main/training/README.md)
- 후속 모델/응용:
  - **SAM2Long**: 긴 비디오에서 occlusion/reappearance 처리 (다중 메모리 뱅크)
  - **SAMURAI**: Kalman filter 기반 motion-aware 메모리 선택
  - **DAM4SAM**: distractor-aware 업데이트 룰
  - **Grounded-SAM 2**: text prompt로 객체 지정

### Citation

```
@article{ravi2024sam2,
  title={SAM 2: Segment Anything in Images and Videos},
  author={Ravi, Nikhila and Gabeur, Valentin and Hu, Yuan-Ting and Hu, Ronghang and
          Ryali, Chaitanya and Ma, Tengyu and Khedr, Haitham and R{\"a}dle, Roman and
          Rolland, Chloe and Gustafson, Laura and Mintun, Eric and Pan, Junting and
          Alwala, Kalyan Vasudev and Carion, Nicolas and Wu, Chao-Yuan and
          Girshick, Ross and Doll{\'a}r, Piotr and Feichtenhofer, Christoph},
  journal={arXiv preprint arXiv:2408.00714},
  year={2024}
}
```

수고하셨습니다! 🎉
